# SatQuery AI — Step 10: Export Model for Local RTX 1650 (4GB) Deployment
### Smart India Hackathon (SIH 26167) | ISRO Space Technology Theme

**Objective**: Take our best fine-tuned remote sensing model from Google Colab and package it for smooth inference on your local PC (Ryzen 5 5500H + 24GB RAM + GTX 1650 4GB GPU).

**What this does**:
1. Merges LoRA weights into the base InternVL3-1B LLM
2. Bundles the trained Sentinel-1 and Sentinel-2 projection heads
3. Formats as a single HuggingFace model folder (~2.2 GB)
4. Creates a zip archive in your Google Drive ready for one-click local download!

In [1]:
# 1. Environment & Drive Setup
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists("/content/SIH"):
    !git clone https://github.com/abhineet115/SIH.git /content/SIH
else:
    !cd /content/SIH && git pull
%cd /content/SIH
!pip install -q -r training/requirements_colab.txt

Mounted at /content/drive
Cloning into '/content/SIH'...
remote: Enumerating objects: 344, done.
remote: Counting objects: 100% (344/344), done.
remote: Compressing objects: 100% (252/252), done.
remote: Total 344 (delta 162), reused 255 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (344/344), 2.43 MiB | 6.37 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/SIH
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.2 MB/s eta 0:00:00


In [2]:
# 2. Run Export Script
import os
drive_root = "/content/drive/MyDrive/SatQuery_AI"
adapter_path = f"{drive_root}/ckpt/r6_fusion/best"
if not os.path.exists(adapter_path):
    print("r6_fusion not found, exporting r2_binary_vqa golden adapter instead...")
    adapter_path = f"{drive_root}/ckpt/r2_binary_vqa/best"

!python training/export.py \
    --base-model OpenGVLab/InternVL3-1B \
    --adapter-path {adapter_path} \
    --output-dir /content/drive/MyDrive/SatQuery_AI/exported_model \
    --quantize-4bit

usage: export.py [-h] --adapter ADAPTER --output OUTPUT
                 [--base-model BASE_MODEL] [--merge-lora] [--no-quantize]
export.py: error: the following arguments are required: --adapter, --output


In [3]:
# 3. Create Zip for Quick Download to Local PC
import shutil
import os

export_dir = "/content/drive/MyDrive/SatQuery_AI/exported_model"
zip_out = "/content/drive/MyDrive/SatQuery_AI/satquery_rs_internvl"

print("Zipping exported model for download...")
shutil.make_archive(zip_out, 'zip', export_dir)
zip_file = f"{zip_out}.zip"
print(f"\n🎉 READY TO DOWNLOAD!")
print(f"File: {zip_file} ({os.path.getsize(zip_file) / (1024*1024):.1f} MB)")
print("Download this file from Google Drive and unzip into: SIH/backend/weights/satquery_rs_internvl/")

Zipping exported model for download...

🎉 READY TO DOWNLOAD!
File: /content/drive/MyDrive/SatQuery_AI/satquery_rs_internvl.zip (0.0 MB)
Download this file from Google Drive and unzip into: SIH/backend/weights/satquery_rs_internvl/


In [4]:
import shutil, os

DRIVE = '/content/drive/MyDrive/SatQuery_AI'
adapter_path = f'{DRIVE}/ckpt/r6_fusion/best'
zip_out = f'{DRIVE}/satquery_rs_internvl'

# Check what's actually there
print("Files in adapter checkpoint:")
for f in os.listdir(adapter_path):
    size_mb = os.path.getsize(f'{adapter_path}/{f}') / (1024*1024)
    print(f"  {f}  ({size_mb:.1f} MB)")

print("\nZipping adapter...")
shutil.make_archive(zip_out, 'zip', adapter_path)
size = os.path.getsize(f'{zip_out}.zip') / (1024*1024)
print(f"\n✅ Zip created: {zip_out}.zip  ({size:.1f} MB)")


Files in adapter checkpoint:
  README.md  (0.0 MB)
  adapter_model.safetensors  (4.1 MB)
  adapter_config.json  (0.0 MB)
  projection_heads.pt  (13.0 MB)
  chat_template.jinja  (0.0 MB)
  tokenizer_config.json  (0.0 MB)
  tokenizer.json  (10.9 MB)

Zipping adapter...

✅ Zip created: /content/drive/MyDrive/SatQuery_AI/satquery_rs_internvl.zip  (17.9 MB)


In [5]:
import shutil

# projection_heads.pt is saved alongside the adapter
for ckpt_name in ['r2_binary_vqa', 'r3a_mcq', 'r3b_captioning', 'r4_grounding', 'r5_change', 'r6_fusion']:
    ph = f'{DRIVE}/ckpt/{ckpt_name}/best/projection_heads.pt'
    if os.path.exists(ph):
        size = os.path.getsize(ph)/(1024*1024)
        print(f"✅ {ckpt_name} projection_heads.pt: {size:.1f} MB")


✅ r2_binary_vqa projection_heads.pt: 13.0 MB
✅ r3a_mcq projection_heads.pt: 13.0 MB
✅ r3b_captioning projection_heads.pt: 13.0 MB
✅ r4_grounding projection_heads.pt: 13.0 MB
✅ r5_change projection_heads.pt: 13.0 MB
✅ r6_fusion projection_heads.pt: 13.0 MB


In [6]:
from google.colab import files
files.download(f'{DRIVE}/satquery_rs_internvl.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>